In [0]:
# Project configuration
PROJECT_NAME = "retail"
VOLUME_PATH = "/Volumes/workspace/retail_bronze/olist_raw"
CATALOG = "workspace"
BRONZE_SCHEMA = f"{PROJECT_NAME}_bronze"

print(f"Project: {PROJECT_NAME}")
print(f"Reading from: {VOLUME_PATH}")
print(f"Writing to: {CATALOG}.{BRONZE_SCHEMA}")

In [0]:
%sql
-- Create all three schemas for the project
CREATE SCHEMA IF NOT EXISTS workspace.retail_bronze
  COMMENT 'Raw data layer - exact copy of source files';

CREATE SCHEMA IF NOT EXISTS workspace.retail_silver
  COMMENT 'Cleaned and transformed data layer';

CREATE SCHEMA IF NOT EXISTS workspace.retail_gold
  COMMENT 'Business-ready aggregated data layer';

-- Verify they exist
SHOW SCHEMAS IN workspace LIKE 'retail*';

In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import StringType

def ingest_csv_to_bronze(file_name: str, table_name: str) -> int:
    
    base_path = "/Volumes/workspace/retail_bronze/olist_raw/"
    full_path = f"{base_path}{file_name}"
    
    print(f"Reading: {full_path}")
    
    # Read CSV with metadata column enabled
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(full_path)
    )
    
    # Add metadata columns (Unity Catalog safe)
    df = (
        df
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    
    row_count = df.count()
    
    # Write to Bronze schema as Delta table
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"workspace.retail_bronze.{table_name}")
    )
    
    print(f"✅ Loaded {row_count:,} rows into workspace.retail_bronze.{table_name}\n")
    
    return row_count


In [0]:
files_to_ingest = {
    "olist_orders_dataset.csv":                "bronze_orders",
    "olist_customers_dataset.csv":             "bronze_customers",
    "olist_order_items_dataset.csv":           "bronze_order_items",
    "olist_order_payments_dataset.csv":        "bronze_order_payments",
    "olist_order_reviews_dataset.csv":         "bronze_order_reviews",
    "olist_products_dataset.csv":              "bronze_products",
    "olist_sellers_dataset.csv":               "bronze_sellers",
    "olist_geolocation_dataset.csv":           "bronze_geolocation",
    "product_category_name_translation.csv":   "bronze_category_translation"
}

print("Starting Bronze Layer Ingestion...\n")
print("=" * 60)

total_rows = 0

for file_name, table_name in files_to_ingest.items():
    rows = ingest_csv_to_bronze(file_name, table_name)
    total_rows += rows

print("=" * 60)
print(f"✅ COMPLETE — Total rows ingested: {total_rows:,}")


In [0]:
%sql
-- Show all bronze tables
SHOW TABLES IN workspace.retail_bronze;

In [0]:
%sql
    
-- Preview the main orders table
SELECT 
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    ingestion_timestamp
FROM workspace.retail_bronze.bronze_orders 
LIMIT 10;

In [0]:
# Get row counts for all bronze tables
print("Bronze Layer Summary\n")
print("=" * 60)

for table_name in files_to_ingest.values():
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    count = spark.table(full_name).count()
    print(f"{table_name:40} {count:>10,} rows")

print("=" * 60)
